Notebook para treinar o professor no Kaggle (ver README, seção Kaggle).

Antes de rodar: em **Settings**, escolha **Accelerator: GPU T4 x2** (ou P100) e ligue **Internet** (para baixar os pesos pré-treinados). Em **Add Input**, anexe os três datasets:
- `andrewmvd/isic-2019` (imagens de treino e validação)
- `lesoes-pele-meta` (rótulos, splits e código; privado)
- `lesoes-pele-extra` (imagens de teste e sem rótulo; privado)

Para treinar, use **Save Version → Save & Run All**: roda em segundo plano (até 12 h) e guarda tudo na aba Output.

# Ambiente

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers, applications, callbacks, optimizers

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

SEED = 42
keras.utils.set_random_seed(SEED)
# teste rápido no PC (poucas imagens, resolução baixa); no Kaggle fica desligado
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
# T4 x2: uma cópia do modelo em cada GPU, cada uma com metade do batch (~2x mais rápido).
# Se der erro ligado a MirroredStrategy/distribute, mude para False e rode com uma GPU.
MULTI_GPU = True

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if gpus:
    # contas em float16 (tensor cores da T4/P100): ~2x mais rápido e metade da memória
    keras.mixed_precision.set_global_policy("mixed_float16")
    use_multi = MULTI_GPU and len(gpus) > 1
    strategy = tf.distribute.MirroredStrategy() if use_multi else tf.distribute.get_strategy()
else:
    print("ATENÇÃO: sem GPU. Em Settings > Accelerator, escolha GPU T4 x2 ou P100.")
    strategy = tf.distribute.get_strategy()
print("réplicas:", strategy.num_replicas_in_sync)

# Dados

In [ ]:
# Os datasets anexados ficam em /kaggle/input, cada um com sua estrutura de pastas;
# as imagens são localizadas pelo nome do arquivo.
INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)

# pasta do dataset lesoes-pele-meta (followlinks: /kaggle/input pode ter links simbólicos)
META = next(Path(d) for d, _, files in os.walk(INPUT, followlinks=True) if "md5.csv" in files)
sys.path.insert(0, str(META))
from prepare_data import CLASSES, HAM_CLASSES
from pipeline import CLASS_IDX, HAM_IDX, N_META, build_cache, locate_images, make_ds

df = locate_images(pd.read_csv(META / "labels.csv", low_memory=False), INPUT)
missing = df[df["image_path"].isna()]
print("imagens por split:", df["split"].value_counts().to_dict())
print("faltando:", missing["split"].value_counts().to_dict())
assert missing.empty, "Faltam imagens: confira se os três datasets estão anexados (ver topo)"

In [ ]:
# Confere por md5 que as imagens são as mesmas da base local (amostra por split)
N_CHECK = 100


def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


expected = pd.read_csv(META / "md5.csv").set_index("file_id")["md5"]
sample = df.groupby("split").sample(N_CHECK, random_state=SEED)
wrong = [f for f, p in zip(sample["file_id"], sample["image_path"]) if md5(p) != expected[f]]
print(f"{len(sample) - len(wrong)}/{len(sample)} imagens idênticas")
assert not wrong, f"Imagens diferentes da base local: {wrong[:5]}"

In [ ]:
# Pré-processamento (Shades of Gray + lado menor de 512 px), feito uma única vez: ~10 min.
# Fica em /tmp, que não entra no output salvo da versão.
CACHE_DIR = Path(os.environ.get("CACHE_DIR", "/tmp/cache_512"))
if SMOKE_TEST:
    df = pd.concat([
        df[df["split"].isin(["train", "val"])].groupby(["split", "dx"]).head(4),
        df[df["split"] == "test"].groupby("dx_ham").head(3),
        df[df["split"] == "unlabeled"].head(8),
    ])
df = build_cache(df, CACHE_DIR)

split = {s: df[df["split"] == s].reset_index(drop=True) for s in ["train", "val", "test", "unlabeled"]}
y_val = split["val"]["dx"].map(CLASS_IDX).values
y_test = split["test"]["dx_ham"].map(HAM_IDX).values

# Prior do treino. O modelo aprende com entropia cruzada comum (probabilidades calibradas, boas
# para a destilação) e, na hora de decidir, subtrai tau * log(prior) dos logits para não
# favorecer as classes grandes como nv (ajuste de logits, Menon et al., 2021).
prior = split["train"]["dx"].value_counts(normalize=True).reindex(CLASSES).values
LOG_PRIOR = np.log(prior)
print({s: len(d) for s, d in split.items()})
print("prior do treino:", {c: round(float(p), 3) for c, p in zip(CLASSES, prior)})

# Modelo professor

Estratégias usadas, todas com resultado comprovado em dermatoscopia (ISIC 2018/2019):

- **Transfer learning**: backbone pré-treinado no ImageNet, em duas fases: primeiro só a cabeça com a rede congelada (é o *baseline* de embeddings, registrado para comparação) e depois o ajuste fino da rede inteira.
- **Resolução alta (384 px)**: detalhes como a rede de pigmento e os vasos somem em 224 px.
- **Constância de cor Shades of Gray**: uniformiza a cor entre os dermatoscópios do HAM, do BCN e do MSK.
- **Data augmentation**: recortes, rotações, espelhamentos, variação de cor e uma **borda circular artificial**, para o modelo não usar a borda preta do BCN como atalho.
- **Metadados** (idade, sexo, região do corpo) como segunda entrada.
- **Desbalanceamento**: entropia cruzada comum + ajuste de logits pelo prior, com o tau escolhido na validação.
- **Otimização**: AdamW, aquecimento + decaimento cosseno do learning rate, float16, 2 GPUs, melhor época pela acurácia balanceada da validação, parada antecipada.
- **Test-time augmentation**: média das previsões em 8 versões da imagem.
- **Ensemble**: média de professores com arquiteturas diferentes (EfficientNetV2, ConvNeXt, EfficientNet), treinados em execuções separadas.

In [ ]:
# Professores disponíveis. O batch é por GPU, com folga para a T4 (16 GB) em float16; se
# faltar memória (erro OOM), reduza o batch. Tempo estimado na T4 x2, com TTA:
# effv2s ~2 h, effb4 ~3 h, effv2m ~4 h, convnext_s ~5 h.
BACKBONES = {
    "effv2s": dict(fn=applications.EfficientNetV2S, size=384, batch=24, lr=1.5e-4),
    "effv2m": dict(fn=applications.EfficientNetV2M, size=384, batch=12, lr=1e-4),
    "convnext_s": dict(fn=applications.ConvNeXtSmall, size=384, batch=16, lr=1e-4),
    "effb4": dict(fn=applications.EfficientNetB4, size=380, batch=16, lr=1.5e-4),
}
# Professores treinados nesta execução (limite do Kaggle: 12 h por execução). Para o ensemble,
# treine os outros em execuções seguintes e anexe o output das anteriores como input (ver Teste).
MODELS = ["effv2s"]
USE_META = True     # idade, sexo e região do corpo como segunda entrada
HEAD_EPOCHS = 2     # fase 1: rede congelada, só a cabeça
EPOCHS = 30         # fase 2: ajuste fino da rede inteira (máximo; para antes se estagnar)
PATIENCE = 6        # épocas sem melhorar a acurácia balanceada da validação antes de parar
WEIGHT_DECAY = 0.02
TTA_VIEWS = 8       # 4 rotações x 2 espelhamentos na previsão

if SMOKE_TEST:
    BACKBONES = {k: {**v, "size": 96, "batch": 8} for k, v in BACKBONES.items()}
    HEAD_EPOCHS, EPOCHS, TTA_VIEWS = 1, 1, 2

In [ ]:
def build_model(name):
    """Backbone pré-treinado + cabeça que junta a imagem e os metadados.

    Devolve o modelo e o backbone (para congelar/descongelar). A saída são logits em float32.
    """
    cfg = BACKBONES[name]
    size = cfg["size"]
    # pooling="avg" entrega um vetor de características (embedding) por imagem; essas redes já
    # trazem a normalização da entrada de 0-255
    base = cfg["fn"](include_top=False, weights="imagenet", input_shape=(size, size, 3), pooling="avg")
    image = keras.Input((size, size, 3), name="image")
    x = layers.Dropout(0.3)(base(image))
    inputs = image
    if USE_META:
        meta = keras.Input((N_META,), name="meta")
        m = layers.Dense(64, activation="relu")(layers.Dropout(0.1)(meta))
        m = layers.BatchNormalization()(m)
        x = layers.Concatenate()([x, m])
        inputs = {"image": image, "meta": meta}
    logits = layers.Dense(len(CLASSES), dtype="float32", name="logits")(x)
    return keras.Model(inputs, logits, name=name), base

# Treino

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class ValMetrics(callbacks.Callback):
    """A cada época: perda, acurácia e acurácia balanceada na validação.

    A acurácia balanceada (média da sensibilidade das classes, métrica oficial do ISIC) usa os
    logits ajustados pelo prior; é ela que escolhe o melhor checkpoint.
    """

    def __init__(self, ds, y):
        super().__init__()
        self.ds, self.y = ds, y

    def on_epoch_end(self, epoch, logs=None):
        logits = self.model.predict(self.ds, verbose=0)
        p = softmax(logits)
        logs["val_loss"] = float(-np.log(p[np.arange(len(self.y)), self.y] + 1e-12).mean())
        logs["val_acc"] = accuracy_score(self.y, logits.argmax(1))
        logs["val_bal_acc"] = balanced_accuracy_score(self.y, (logits - LOG_PRIOR).argmax(1))
        print(f"  época {epoch + 1}: val_loss={logs['val_loss']:.4f} "
              f"val_acc={logs['val_acc']:.4f} val_bal_acc={logs['val_bal_acc']:.4f}")


def train_teacher(name):
    cfg = BACKBONES[name]
    size, batch = cfg["size"], cfg["batch"] * strategy.num_replicas_in_sync
    out = WORK / name
    out.mkdir(parents=True, exist_ok=True)
    train_ds = make_ds(split["train"], size, batch, training=True, use_meta=USE_META)
    val_ds = make_ds(split["val"], size, batch, labels=None, use_meta=USE_META)
    loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    # Fase 1: backbone congelado, treina só a cabeça. É o baseline "embeddings pré-treinados
    # + classificador", guardado para comparação.
    with strategy.scope():
        model, base = build_model(name)
        base.trainable = False
        model.compile(optimizer=optimizers.AdamW(1e-3, weight_decay=WEIGHT_DECAY), loss=loss)
    model.fit(train_ds, epochs=HEAD_EPOCHS, verbose=2, callbacks=[
        ValMetrics(val_ds, y_val), callbacks.CSVLogger(str(out / "historico_fase1.csv"))])
    baseline = pd.read_csv(out / "historico_fase1.csv").iloc[-1].to_dict()

    # Fase 2: ajuste fino da rede inteira; 1 época de aquecimento e decaimento cosseno do lr
    per_epoch = int(train_ds.cardinality())
    with strategy.scope():
        base.trainable = True
        schedule = optimizers.schedules.CosineDecay(
            cfg["lr"] / 20, decay_steps=max(1, per_epoch * (EPOCHS - 1)), alpha=0.01,
            warmup_target=cfg["lr"], warmup_steps=per_epoch)
        opt = optimizers.AdamW(schedule, weight_decay=WEIGHT_DECAY)
        opt.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])  # bias e normalizações
        model.compile(optimizer=opt, loss=loss)
    ckpt = out / "melhor.keras"
    model.fit(train_ds, epochs=EPOCHS, verbose=2, callbacks=[
        ValMetrics(val_ds, y_val),
        callbacks.ModelCheckpoint(str(ckpt), monitor="val_bal_acc", mode="max", save_best_only=True),
        callbacks.EarlyStopping(monitor="val_bal_acc", mode="max", patience=PATIENCE),
        callbacks.CSVLogger(str(out / "historico.csv")),
    ])
    with strategy.scope():
        model = keras.models.load_model(ckpt)
    return model, baseline


def tta_view(x, k):
    """Uma das 8 versões da imagem (4 rotações x espelhamento) do test-time augmentation."""
    img = x["image"] if USE_META else x
    img = tf.image.rot90(img, k % 4)
    if k >= 4:
        img = tf.image.flip_left_right(img)
    return {**x, "image": img} if USE_META else img


def predict_tta(model, data, size, batch):
    """Probabilidades médias nas TTA_VIEWS versões de cada imagem."""
    ds = make_ds(data, size, batch, labels=None, use_meta=USE_META)
    probs = 0
    for k in range(TTA_VIEWS):
        probs = probs + softmax(model.predict(ds.map(lambda x, k=k: tta_view(x, k)), verbose=0))
    return probs / TTA_VIEWS

In [ ]:
for name in MODELS:
    print(f"===== {name} =====")
    model, baseline = train_teacher(name)
    cfg = BACKBONES[name]
    batch = cfg["batch"] * strategy.num_replicas_in_sync
    # previsões com TTA de todos os splits: teste, validação e as que o aluno vai imitar
    probs = {s: predict_tta(model, d, cfg["size"], batch) for s, d in split.items()}
    ids = {f"ids_{s}": d["file_id"].to_numpy(dtype=str) for s, d in split.items()}
    np.savez(WORK / name / "probs.npz", **probs, **ids)
    with open(WORK / name / "config.json", "w") as f:
        json.dump({"backbone": name, **{k: v for k, v in cfg.items() if k != "fn"},
                   "use_meta": USE_META, "baseline_fase1": baseline}, f, indent=2, default=float)
    del model
    keras.backend.clear_session()

In [ ]:
# Curvas de treino (a linha tracejada separa a fase 1, rede congelada, da fase 2)
for name in MODELS:
    h = pd.concat([pd.read_csv(WORK / name / "historico_fase1.csv"),
                   pd.read_csv(WORK / name / "historico.csv")], ignore_index=True)
    epoch = np.arange(1, len(h) + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
    a1.plot(epoch, h["loss"], color="#2a78d6", lw=2, label="treino")
    a1.plot(epoch, h["val_loss"], color="#eb6834", lw=2, label="validação")
    a1.set(title=f"{name}: perda", xlabel="época")
    a1.legend(frameon=False)
    a2.plot(epoch, h["val_bal_acc"], color="#2a78d6", lw=2)
    a2.set(title=f"{name}: acurácia balanceada na validação", xlabel="época")
    for a in (a1, a2):
        a.axvline(HEAD_EPOCHS + 0.5, color="#999", ls="--", lw=1)
        a.spines[["top", "right"]].set_visible(False)
        a.grid(axis="y", color="#e5e5e5")
        a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

# Teste

In [ ]:
def find_runs():
    """Previsões desta execução e de execuções anteriores anexadas como input (ensemble)."""
    runs = {}
    for root in (WORK, INPUT):
        for folder, _, files in os.walk(root, followlinks=True):
            if "probs.npz" in files:
                runs.setdefault(Path(folder).name, dict(np.load(Path(folder) / "probs.npz")))
    return runs


runs = find_runs()
for name, r in runs.items():  # todas as execuções precisam ter as mesmas imagens, na mesma ordem
    for s, d in split.items():
        assert (r[f"ids_{s}"] == d["file_id"].to_numpy(dtype=str)).all(), f"{name}: imagens de {s} diferentes"
print("professores:", list(runs))

MALIGNANT_8 = [CLASS_IDX[c] for c in ["ak", "bcc", "mel", "scc"]]
MALIGNANT_7 = [HAM_IDX[c] for c in ["akiec", "bcc", "mel"]]


def adjust(p, tau):
    """Ajuste de logits: divide as probabilidades por prior**tau e renormaliza."""
    return softmax(np.log(p + 1e-12) - tau * LOG_PRIOR)


def to_ham(p):
    """8 classes -> 7 do HAM10000 (akiec = ak + scc), para comparar com o gabarito do teste."""
    return np.stack([p[:, CLASS_IDX["ak"]] + p[:, CLASS_IDX["scc"]] if c == "akiec" else p[:, CLASS_IDX[c]]
                     for c in HAM_CLASSES], axis=1)


def tune_tau(p_val):
    """Escolhe tau na validação (nunca no teste), maximizando a acurácia balanceada."""
    taus = np.arange(0, 2.01, 0.05)
    scores = [balanced_accuracy_score(y_val, adjust(p_val, t).argmax(1)) for t in taus]
    return float(taus[int(np.argmax(scores))])


def evaluate(p_val, p_test):
    tau = tune_tau(p_val)
    raw, adj = to_ham(p_test), to_ham(adjust(p_test, tau))
    return {
        "tau": tau,
        "val_bal_acc": balanced_accuracy_score(y_val, adjust(p_val, tau).argmax(1)),
        # acurácia comum: decisão sem ajuste (é o que maximiza acertos totais)
        "test_acc": accuracy_score(y_test, raw.argmax(1)),
        # acurácia balanceada: decisão ajustada (métrica oficial do ISIC)
        "test_bal_acc": balanced_accuracy_score(y_test, adj.argmax(1)),
        "test_acc_ajustada": accuracy_score(y_test, adj.argmax(1)),
        "test_auc_macro": roc_auc_score(y_test, raw, multi_class="ovr", labels=range(len(HAM_CLASSES))),
        "test_auc_maligno": roc_auc_score(np.isin(y_test, MALIGNANT_7), raw[:, MALIGNANT_7].sum(1)),
    }


results = {name: evaluate(r["val"], r["test"]) for name, r in runs.items()}
p_ens = {s: np.mean([r[s] for r in runs.values()], axis=0) for s in split}
if len(runs) > 1:
    results["ensemble"] = evaluate(p_ens["val"], p_ens["test"])
best = max(results, key=lambda k: results[k]["val_bal_acc"])  # escolhido pela validação
print("melhor na validação:", best)
pd.DataFrame(results).T.round(4)

In [ ]:
# Relatório por classe e matriz de confusão do melhor (decisão ajustada)
p_best = {s: (p_ens if best == "ensemble" else runs[best])[s] for s in split}
p_test = to_ham(adjust(p_best["test"], results[best]["tau"]))
pred = p_test.argmax(1)
labels = list(range(len(HAM_CLASSES)))
print(classification_report(y_test, pred, labels=labels, target_names=HAM_CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(y_test, pred, labels=labels, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False,
            xticklabels=HAM_CLASSES, yticklabels=HAM_CLASSES, ax=ax)
ax.set(xlabel="Previsto", ylabel="Real", title=f"Teste ({best}): fração de cada classe real")
plt.tight_layout()
plt.show()

In [ ]:
# Triagem "maligno ou não" (mel, bcc, akiec). O limiar é escolhido na validação para
# sensibilidade de 95% (deixar passar um câncer é o erro mais grave) e só então aplicado no teste.
score_val = p_best["val"][:, MALIGNANT_8].sum(1)
thr = np.quantile(score_val[np.isin(y_val, MALIGNANT_8)], 0.05)
score_test = to_ham(p_best["test"])[:, MALIGNANT_7].sum(1)
mal_test = np.isin(y_test, MALIGNANT_7)
print(f"limiar (da validação): {thr:.3f}")
print(f"teste: sensibilidade {(score_test[mal_test] >= thr).mean():.3f}, "
      f"especificidade {(score_test[~mal_test] < thr).mean():.3f}, "
      f"AUC {roc_auc_score(mal_test, score_test):.3f}")

In [ ]:
# Comparação com dermatologistas nas mesmas imagens do teste (Tschandl et al., Nature Medicine
# 2020): cada linha do CSV é um diagnóstico de um médico, sem e com o apoio da IA do estudo.
readers_csv = META / "ISIC2018_Task3_Test_NatureMedicine_AI_Interaction_Benefit.csv"
if readers_csv.exists():
    nm = pd.read_csv(readers_csv, decimal=",")
    truth = split["test"].set_index("image_id")["dx_ham"].map(HAM_IDX)
    nm = nm[nm["image_id"].isin(truth.index)]
    y_nm = nm["image_id"].map(truth).values
    ours = pd.Series(pred, index=split["test"]["image_id"].values)

    def answers(prefix):
        return nm[[f"{prefix}_{c}" for c in HAM_CLASSES]].values.argmax(1)

    comp = {
        "Dermatologistas sem IA": answers("user_dx_without_interaction"),
        "Dermatologistas com a IA do estudo": answers("user_dx_with_interaction"),
        "IA do estudo": answers("prob_m_dx"),
        f"Nosso professor ({best})": nm["image_id"].map(ours).values,
    }
    print(f"{len(nm)} diagnósticos em {nm['image_id'].nunique()} imagens do teste")
    display(pd.DataFrame({k: {"acurácia": accuracy_score(y_nm, v),
                              "acurácia balanceada": balanced_accuracy_score(y_nm, v)}
                          for k, v in comp.items()}).T.round(3))

# Exportação

In [ ]:
# Tudo fica em /kaggle/working e aparece na aba Output da versão salva:
#   <professor>/melhor.keras     modelo treinado (melhor época na validação)
#   <professor>/probs.npz        probabilidades com TTA de train/val/test/unlabeled
#   <professor>/historico*.csv   curvas de treino;  <professor>/config.json  configuração
#   ensemble_probs.npz           probabilidades do melhor (professor ou ensemble), para o aluno
#   resultados.csv               tabela de métricas
# Atenção: no split train, o professor viu essas imagens no treino, então as probabilidades são
# mais confiantes do que seriam em imagens novas; vale considerar isso na destilação.
np.savez(WORK / "ensemble_probs.npz", **p_best, tau=results[best]["tau"], modelo=best,
         **{f"ids_{s}": d["file_id"].to_numpy(dtype=str) for s, d in split.items()})
pd.DataFrame(results).T.to_csv(WORK / "resultados.csv")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(WORK)}  {p.stat().st_size / 1e6:.1f} MB")

# Registro de iterações